# Módulo 1 — Segmentación de Clientes con K-Means
### Proyecto: Segmentación de Clientes en Centros Comerciales
**Materia:** Ciencia de Datos I  
**Institución:** ETITC  
**Autores:** Daniel Valencia, Daniel Medcalfe

## 1. Importar librerías

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
import os

# Ajusta esta ruta a donde está tu proyecto
os.chdir(r'C:\Users\Admin\Documents\segmentacion_de_clientes')
warnings.filterwarnings('ignore')

## 2. Cargar datos limpios

In [11]:
df = pd.read_csv('data/processed/datos_limpios.csv')
df['invoice_date'] = pd.to_datetime(df['invoice_date'])

print("Filas:", len(df))
print("Columnas:", df.columns.tolist())
df.head(3)

Filas: 99457
Columnas: ['invoice_no', 'customer_id', 'gender', 'age', 'category', 'quantity', 'price', 'payment_method', 'invoice_date', 'shopping_mall', 'total_spend', 'year', 'month']


,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall,total_spend,year,month
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,2022-08-05,Kanyon,7502.00,2022,8
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,2021-12-12,Forum Istanbul,5401.53,2021,12
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,2021-11-09,Metrocity,300.08,2021,11


## 3. Preparar variables para el clustering

Usamos estas variables como perfil de cada transacción:
- **age**: edad del cliente
- **total_spend**: gasto total
- **quantity**: cantidad de productos
- **category_code**: tipo de producto (codificado)
- **gender_code**: género (0 = Femenino, 1 = Masculino)

> Nota: cada customer_id aparece una sola vez en el dataset, por eso usamos variables de la transacción como proxy del perfil del cliente.

In [12]:
# Codificar variables categóricas
df['category_code'] = df['category'].astype('category').cat.codes
df['gender_code']   = (df['gender'] == 'Male').astype(int)

# Seleccionar features
features = ['age', 'total_spend', 'quantity', 'category_code', 'gender_code']
X = df[features].copy()

# Escalar (importante para K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Variables seleccionadas:", features)
print("Shape de la matriz:", X_scaled.shape)

Variables seleccionadas: ['age', 'total_spend', 'quantity', 'category_code', 'gender_code']
Shape de la matriz: (99457, 5)


## 4. Método del Codo — elegir el número de clusters

Probamos K de 2 a 10 y vemos cuándo la inercia deja de bajar mucho.  
También revisamos el Silhouette Score (objetivo: mayor a 0.40).

In [ ]:
inercias    = []
silhouettes = []
k_valores   = range(2, 11)

for k in k_valores:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))
    print(f"k={k}  inercia={km.inertia_:,.0f}  silhouette={silhouette_score(X_scaled, km.labels_):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_valores, inercias, marker='o', color='steelblue')
axes[0].set_title('Método del Codo — Inercia')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia')

axes[1].plot(k_valores, silhouettes, marker='o', color='darkorange')
axes[1].axhline(0.40, color='red', linestyle='--', label='Objetivo 0.40')
axes[1].set_title('Silhouette Score por k')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/kmeans_codo.png')
plt.show()

## 5. Entrenar el modelo con k elegido

In [ ]:
# Cambiar este valor según las gráficas del método del codo
K = 4

modelo = KMeans(n_clusters=K, random_state=42, n_init=10)
df['cluster'] = modelo.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, df['cluster'])
print(f"Clusters: {K}")
print(f"Silhouette Score: {sil:.4f}")
print(f"Resultado: {'✓ cumple objetivo' if sil >= 0.40 else '✗ por debajo de 0.40'}")
print()
print("Clientes por cluster:")
print(df['cluster'].value_counts().sort_index())

## 6. Visualizar clusters en 2D con PCA

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)

df['pca_x'] = coords[:, 0]
df['pca_y'] = coords[:, 1]

colores = ['steelblue', 'darkorange', 'green', 'crimson',
           'purple', 'brown', 'pink', 'gray']

plt.figure(figsize=(9, 6))
for c in sorted(df['cluster'].unique()):
    sub = df[df['cluster'] == c]
    plt.scatter(sub['pca_x'], sub['pca_y'],
                label=f'Cluster {c+1}  (n={len(sub):,})',
                color=colores[c], alpha=0.5, s=12)

plt.title(f'Clusters K-Means en 2D (PCA) — k={K}, Silhouette={sil:.3f}')
plt.xlabel('Componente 1')
plt.ylabel('Componente 2')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/kmeans_pca.png')
plt.show()

## 7. Perfil de cada segmento

In [ ]:
resumen = df.groupby('cluster').agg(
    n              = ('age', 'count'),
    edad_promedio  = ('age', 'mean'),
    gasto_promedio = ('total_spend', 'mean'),
    cantidad_prom  = ('quantity', 'mean'),
    categoria_top  = ('category', lambda x: x.value_counts().index[0]),
    pct_femenino   = ('gender', lambda x: (x == 'Female').mean() * 100),
    pago_frecuente = ('payment_method', lambda x: x.value_counts().index[0]),
).round(2)

resumen.index = [f'Cluster {i+1}' for i in resumen.index]
print(resumen.to_string())

In [ ]:
# Gráfica comparativa de gasto por cluster
gasto = df.groupby('cluster')['total_spend'].mean()

plt.figure(figsize=(8, 4))
gasto.plot(kind='bar', color=colores[:K], edgecolor='white')
plt.title('Gasto promedio por cluster')
plt.ylabel('Gasto promedio (USD)')
plt.xticks([i for i in range(K)],
           [f'Cluster {i+1}' for i in range(K)], rotation=0)
plt.tight_layout()
plt.savefig('../reports/figures/kmeans_perfiles.png')
plt.show()

## 8. Guardar resultados

In [ ]:
# Exportar dataset con etiqueta de cluster
cols = ['invoice_no', 'customer_id', 'age', 'gender', 'category',
        'quantity', 'price', 'total_spend', 'payment_method',
        'shopping_mall', 'cluster']

df[cols].to_csv('../reports/resultados/clientes_segmentados.csv', index=False)

print("Archivo guardado: reports/resultados/clientes_segmentados.csv")
print("Gráficas guardadas en: reports/figures/")